# Telefon Bankacılığı için Metin Sınıflandırma (NLP)

Müşterinin yazdığı cümleyi **11 işlem kategorisinden birine** atayan bir makine öğrenmesi modeli.

Bu not defteri, hazır bir başlangıç koduna her seferinde **tek bir değişiklik** yapıp skorun nasıl değiştiğini ölçerek öğrendiklerimin kaydıdır.

**Kategoriler:** Altın Döviz, BES, EFT FAST, Hisse Senedi Fon, Kart Kayıp, Kart Şifresi Değiştirme, Kredi Başvurusu, Kredi Kartı Borcu, Müşteri Temsilcisi, Swift ve Yurtdışı, İnternete açma

**Veri:** `banka.csv`, iki sütun: `metin` (müşteri cümlesi) ve `kategori` (doğru işlem).

**Akış:** Veriyi oku, mesaj al, temizle, sayıya çevir, böl, modeli eğit, ölç, tahmin et.

**Çalıştırma:** Hücre `input()` ile bir mesaj sorar. Mesajını yaz ve Enter'a bas.

## 1. Kütüphaneler

- `pandas`, `numpy`: veri tablosu ve sayısal işlemler
- `CountVectorizer`: metni sayıya çevirir
- `LogisticRegression`, `RandomForestClassifier`: sınıflandırma modelleri
- `train_test_split`: veriyi eğitim ve test olarak ayırır
- `classification_report`: kategori bazında başarı raporu

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

## 2. Veri, model ve değerlendirme

Aşağıdaki hücre adım adım şunları yapar:

1. **Veriyi okur** ve stopword listesini tanımlar.
2. **Kullanıcıdan mesaj alır.** Mesaj tablonun en altına geçici etiketle (`0`) eklenir, böylece eğitim cümleleriyle aynı işlemlerden geçer.
3. **Stopword'leri temizler** (anlam taşımayan "ve", "ile", "için" gibi kelimeler).
4. **Metni sayıya çevirir** (`CountVectorizer`). Her cümle, sözlükteki kelimelerin kaç kez geçtiğini gösteren bir sayı satırı olur.
5. **Eğitim ve test olarak böler.** `stratify=y` her kategorinin iki tarafta aynı oranda olmasını sağlar, `random_state=21` bölmeyi her çalıştırmada aynı yapar.
6. **Modeli eğitir** ve test verisiyle skorunu ölçer.
7. **Kullanıcının mesajını tahmin eder.**
8. **Kategori bazında rapor** basar.

In [ ]:

df=pd.read_csv("banka.csv")
stopwords=['fakat','lakin','ancak','acaba', 'ama', 'aslında', 'az', 'bazı', 'belki', 'biri', 'birkaç', 'birşey', 'biz', 'bu', 'çok', 'çünkü', 'da', 'daha', 'de', 'defa', 'diye', 'eğer', 'en', 'gibi', 'hem', 'hep', 'hepsi', 'her', 'hiç', 'için', 'ile', 'ise', 'kez', 'ki', 'kim', 'mı', 'mu', 'mü', 'nasıl', 'ne', 'neden', 'nerde', 'nerede', 'nereye', 'niçin', 'niye', 'o', 'sanki', 'şey', 'siz', 'şu', 'tüm', 've', 'veya', 'ya', 'yani']

mesaj=input("Yapmak istediginiz işlemi Giriniz.")

mesajdf=pd.DataFrame({"metin":[mesaj],"kategori":[0]},index=[42])
df=pd.concat([df,mesajdf],ignore_index=True)


for word in stopwords:
  word=" "+word+" "
  df["metin"]=df["metin"].str.replace(word," ")

cv=CountVectorizer()

x=cv.fit_transform(df["metin"]).toarray()
y=df["kategori"]

tahmin=x[-1].copy()
x=x[0:-1]
y=y[0:-1]


x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=21,train_size=0.7,stratify=y)

#rf=RandomForestClassifier(random_state=21)   # önceki model: Random Forest (50 kelimelik sözlükle skor 0.72)
rf=LogisticRegression(max_iter=1000)
model=rf.fit(x_train,y_train)
skor=model.score(x_test,y_test)

sonuc=model.predict([tahmin])

print("Sonuc:",sonuc,"Skor:",skor)

y_pred=model.predict(x_test)
print(classification_report(y_test,y_pred))

Yapmak istediginiz işlemi Giriniz.kartımı kaybettim
Sonuc: ['Kart Kayıp'] Skor: 0.9255952380952381
                         precision    recall  f1-score   support

            Altın Döviz       0.95      0.88      0.92        43
                    BES       0.95      0.92      0.93        60
               EFT FAST       0.84      0.85      0.84        86
       Hisse Senedi Fon       0.98      0.90      0.94        70
             Kart Kayıp       0.97      0.95      0.96        37
Kart Şifresi Değiştirme       1.00      1.00      1.00        50
        Kredi Başvurusu       0.96      0.99      0.97        74
      Kredi Kartı Borcu       0.96      0.92      0.94        50
     Müşteri Temsilcisi       0.91      0.88      0.89        67
      Swift ve Yurtdışı       0.81      0.95      0.87        77
         İnternete açma       0.98      0.98      0.98        58

               accuracy                           0.93       672
              macro avg       0.94      0.93      0.93

### Sonuç

- **Genel doğruluk (accuracy): 0.93** (`Skor: 0.9256`)
- Örnek mesaj `kartımı kaybettim` için model **Kart Kayıp** kategorisini buldu.
- En iyi kategori: `Kart Şifresi Değiştirme` (f1 = 1.00).
- En düşük f1: `EFT FAST` (0.84). `Swift ve Yurtdışı` kategorisinde precision 0.81 ile en düşük, yani model bu kategoriyi bazen fazla iddialı tahmin ediyor.

**Rapordaki ölçüler**

| Ölçü | Sorduğu soru |
|---|---|
| precision | Model bir kategoriyi söylediğinde ne kadar haklı? |
| recall | O kategoriye gerçekten ait olanların kaçını buldu? |
| f1-score | İkisinin dengesi |
| support | Test setinde o kategoriden kaç cümle var? |

## 3. Eğitim ve test oranları kontrolü

`stratify=y` çalıştı mı? Her kategorinin oranı eğitimde ve testte birbirine çok yakın olmalı. Aşağıdaki ilk liste eğitim, ikinci liste test oranlarını gösterir (örneğin `Kart Kayıp`: yaklaşık 0.054 ve 0.055).

In [ ]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

kategori
EFT FAST                   0.128827
Swift ve Yurtdışı          0.114158
Kredi Başvurusu            0.110332
Hisse Senedi Fon           0.104592
Müşteri Temsilcisi         0.099490
BES                        0.089923
İnternete açma             0.085459
Kart Şifresi Değiştirme    0.073980
Kredi Kartı Borcu          0.073980
Altın Döviz                0.065051
Kart Kayıp                 0.054209
Name: proportion, dtype: float64
kategori
EFT FAST                   0.127976
Swift ve Yurtdışı          0.114583
Kredi Başvurusu            0.110119
Hisse Senedi Fon           0.104167
Müşteri Temsilcisi         0.099702
BES                        0.089286
İnternete açma             0.086310
Kart Şifresi Değiştirme    0.074405
Kredi Kartı Borcu          0.074405
Altın Döviz                0.063988
Kart Kayıp                 0.055060
Name: proportion, dtype: float64


## 4. Modelin gördüğü sayı satırı

Kullanıcının mesajının sayıya çevrilmiş hali. Her cümle, sözlükteki kelime sayısı kadar sayıdan oluşan bir satıra dönüşür (burada 1874 sayı). Çoğu sıfırdır, çünkü bir cümle sözlükteki kelimelerin çok azını içerir.

In [ ]:
tahmin


array([0, 0, 0, ..., 0, 0, 0], shape=(1874,))

## 5. Kategoriler

Tablodaki benzersiz kategoriler. Sondaki `0`, kullanıcının mesajı için eklediğimiz geçici etikettir, gerçek bir kategori değildir.

In [ ]:
df["kategori"].unique()

array(['Kart Şifresi Değiştirme', 'Kredi Kartı Borcu',
       'Müşteri Temsilcisi', 'BES', 'Kart Kayıp', 'Hisse Senedi Fon',
       'Altın Döviz', 'EFT FAST', 'Swift ve Yurtdışı', 'İnternete açma',
       'Kredi Başvurusu', 0], dtype=object)

## Deney tablosu

Her seferinde yalnızca bir şeyi değiştirdim ve skoru not ettim.

| Deney | Model ve sözlük | Skor |
|---|---|---|
| Başlangıç | Random Forest, en sık 50 kelimelik sözlük (`max_features=50`) | 0.72 |
| Son durum | Lojistik Regresyon, sınırsız sözlük (`CountVectorizer()`) | 0.93 |

Sözlük sınırını kaldırmak ve Lojistik Regresyon'a geçmek birlikte skoru 0.72'den 0.93'e çıkardı.

## Öğrendiklerim

- **`random_state`:** Rastgele işlemleri sabitler, sonuç her çalıştırmada aynı çıkar. Skoru yükseltmez.
- **`stratify`:** Eğitim ve testte her kategorinin oranını korur.
- **`classification_report`:** Genel skorun göstermediği kategori bazında zayıflıkları gösterir.
- **Sözlük boyutu:** Sözlük çok küçük olunca model kategoriyi ayıran kelimelerin çoğunu göremiyor.
- **Model karşılaştırması:** Aynı veriyle farklı modeller denemek, hangisinin daha iyi çalıştığını deneyle gösterir.